# 📊 CloudSync SaaS Analytics: 01. Data Cleaning & Integrity Audit

**Author:** Antigravity Data Intelligence & Business Analytics Team  
**Dataset:** CloudSync Enterprise Customer & Subscription Ecosystem (12,000+ Accounts)  
**Objective:** Perform enterprise-grade data auditing, missing value treatment, deduplication, casing standardization, date validation, outlier remediation, and export clean star-schema datasets.

---

### Business Context
Data quality directly impacts strategic retention decisions and ML reliability. Prior to modeling customer lifetime behavior and churn drivers, we execute systematic data hygiene protocols across all 6 operational data streams:
1. `customers.csv` — Firmographic and acquisition demographic profiles
2. `subscriptions.csv` — Billing plans, contract types, terms, MRR, and renewal flags
3. `customer_activity.csv` — High-frequency monthly product engagement telemetry
4. `support_tickets.csv` — Support desk tickets, resolution time, priority, and ticket CSAT
5. `payments.csv` — Transaction histories, payment gateways, and delinquency metrics
6. `customer_feedback.csv` — Net Promoter Score (NPS) surveys and qualitative verbatim

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Display configuration
pd.set_option('display.max_columns', 25)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

DATA_DIR = '../data'
CLEAN_DIR = '../data/cleaned'
os.makedirs(CLEAN_DIR, exist_ok=True)

print("Libraries imported and paths configured.")

## 1. Data Ingestion & Shape Inspection

In [ ]:
df_cust = pd.read_csv(os.path.join(DATA_DIR, "customers.csv"))
df_subs = pd.read_csv(os.path.join(DATA_DIR, "subscriptions.csv"))
df_act = pd.read_csv(os.path.join(DATA_DIR, "customer_activity.csv"))
df_tickets = pd.read_csv(os.path.join(DATA_DIR, "support_tickets.csv"))
df_payments = pd.read_csv(os.path.join(DATA_DIR, "payments.csv"))
df_fb = pd.read_csv(os.path.join(DATA_DIR, "customer_feedback.csv"))

print(f"Customers shape:        {df_cust.shape}")
print(f"Subscriptions shape:    {df_subs.shape}")
print(f"Activity records shape: {df_act.shape}")
print(f"Support tickets shape:  {df_tickets.shape}")
print(f"Payments records shape: {df_payments.shape}")
print(f"Feedback records shape: {df_fb.shape}")

## 2. Customer Dimension: Deduplication & Text Normalization

In [ ]:
# 1. Check duplicates
dup_cust_count = df_cust.duplicated(subset=['customer_id']).sum()
print(f"Duplicate customer records found: {dup_cust_count}")

# Remove duplicates
df_cust_clean = df_cust.drop_duplicates(subset=['customer_id']).copy()

# 2. Text Normalization: Fix mixed casing in customer_name
df_cust_clean['customer_name'] = df_cust_clean['customer_name'].astype(str).str.title().str.strip()

# 3. Missing Value Analysis
missing_cust = df_cust_clean.isnull().sum()
print("
Missing values in Customers dimension:")
print(missing_cust[missing_cust > 0])

# Impute missing categorical values
df_cust_clean['city'] = df_cust_clean['city'].fillna('Unknown')
df_cust_clean['state'] = df_cust_clean['state'].fillna('Unknown')
df_cust_clean['acquisition_channel'] = df_cust_clean['acquisition_channel'].fillna('Direct/Unknown')

# Data Type Enforcement
df_cust_clean['signup_date'] = pd.to_datetime(df_cust_clean['signup_date'])
df_cust_clean['age'] = df_cust_clean['age'].astype(int)

print(f"
Cleaned Customers count: {len(df_cust_clean)}")
df_cust_clean.head(5)

## 3. Subscriptions Fact: Pricing, Dates & Contract Integrity

In [ ]:
df_subs_clean = df_subs.copy()

# Verify missing subscription status or anomalies
print("Subscription status distribution:")
print(df_subs_clean['subscription_status'].value_counts())

# Date parsing
df_subs_clean['subscription_start_date'] = pd.to_datetime(df_subs_clean['subscription_start_date'])
df_subs_clean['subscription_end_date'] = pd.to_datetime(df_subs_clean['subscription_end_date'])

# Numeric types
df_subs_clean['monthly_fee'] = df_subs_clean['monthly_fee'].astype(float)
df_subs_clean['discount_percentage'] = df_subs_clean['discount_percentage'].astype(float)

# Validate logical consistency: End date must be greater than start date for churned users
invalid_dates = df_subs_clean[
    (df_subs_clean['subscription_end_date'].notnull()) & 
    (df_subs_clean['subscription_end_date'] < df_subs_clean['subscription_start_date'])
]
print(f"Inconsistent subscription date records: {len(invalid_dates)}")

df_subs_clean.head(5)

## 4. Support Tickets: Date Formatting & Outlier Remediation

In [ ]:
df_tickets_clean = df_tickets.copy()

# Standardize mixed date strings
df_tickets_clean['ticket_date'] = pd.to_datetime(df_tickets_clean['ticket_date'], format='mixed')

# Outlier Analysis on resolution_time_hours
print("Resolution Time (Hours) Summary Statistics:")
print(df_tickets_clean['resolution_time_hours'].describe())

# Remediation of sensor/telemetry error (999.9 hours)
outlier_count = (df_tickets_clean['resolution_time_hours'] > 200).sum()
median_res = df_tickets_clean[df_tickets_clean['resolution_time_hours'] < 200]['resolution_time_hours'].median()
print(f"
Remediating {outlier_count} resolution time outliers with category-median ({median_res:.1f} hours)...")

df_tickets_clean.loc[df_tickets_clean['resolution_time_hours'] > 200, 'resolution_time_hours'] = median_res
df_tickets_clean['satisfaction_score'] = df_tickets_clean['satisfaction_score'].astype(int)

df_tickets_clean.head(5)

## 5. Payments Fact: Negative Amounts & Extreme Outliers Treatment

In [ ]:
df_payments_clean = df_payments.copy()

# Outlier & negative amount detection
print("Payment Amount Summary Statistics:")
print(df_payments_clean['amount'].describe())

# Fix negative values (erroneous sign reversals)
df_payments_clean['amount'] = df_payments_clean['amount'].abs()

# Cap unrealistic outlier payments (> $15,000) to standard enterprise annual max ($5,988)
outlier_pay = (df_payments_clean['amount'] > 15000).sum()
print(f"Remediating {outlier_pay} extreme payment anomalies...")
df_payments_clean.loc[df_payments_clean['amount'] > 15000, 'amount'] = 5988.0

df_payments_clean['payment_date'] = pd.to_datetime(df_payments_clean['payment_date'])
df_payments_clean['days_overdue'] = df_payments_clean['days_overdue'].astype(int)

print(f"Cleaned Payments count: {len(df_payments_clean)}")
df_payments_clean.head(5)

## 6. Feedback & Product Activity Cleansing

In [ ]:
# 1. Product Activity
df_act_clean = df_act.copy()
df_act_clean['activity_month'] = pd.to_datetime(df_act_clean['activity_month'] + '-01')
for c in ['login_count', 'active_days', 'session_count', 'features_used', 'projects_created', 'files_uploaded', 'api_usage_count']:
    df_act_clean[c] = df_act_clean[c].astype(int)
df_act_clean['avg_session_minutes'] = df_act_clean['avg_session_minutes'].astype(float)

# 2. Feedback
df_fb_clean = df_fb.copy()
df_fb_clean['feedback_date'] = pd.to_datetime(df_fb_clean['feedback_date'])
df_fb_clean['nps_score'] = df_fb_clean['nps_score'].astype(int)
df_fb_clean['satisfaction_score'] = df_fb_clean['satisfaction_score'].astype(int)

print("Activity and Feedback cleaned successfully.")

## 7. Export Clean Datasets to Production Storage

In [ ]:
# Export clean CSVs
df_cust_clean.to_csv(os.path.join(CLEAN_DIR, "customers_clean.csv"), index=False)
df_subs_clean.to_csv(os.path.join(CLEAN_DIR, "subscriptions_clean.csv"), index=False)
df_act_clean.to_csv(os.path.join(CLEAN_DIR, "customer_activity_clean.csv"), index=False)
df_tickets_clean.to_csv(os.path.join(CLEAN_DIR, "support_tickets_clean.csv"), index=False)
df_payments_clean.to_csv(os.path.join(CLEAN_DIR, "payments_clean.csv"), index=False)
df_fb_clean.to_csv(os.path.join(CLEAN_DIR, "customer_feedback_clean.csv"), index=False)

print("All 6 cleaned production datasets successfully written to data/cleaned/.")